## 6.3 Simple RNN反向传播 - 示例

#### 1、为什么要做这个 3 步示例？

##### 1.1 前面我们已经分别学习了：

- RNN 的前向传播
- RNN 的损失函数
- RNN 的 BPTT 反向传播路径和公式

所以这一小节我们要做的事情就是：

用一个总时间步 $T=3$ 的小序列，把整套流程完整串起来。

也就是从下面这条主线来理解：

$x_1, x_2, x_3 \rightarrow h_1, h_2, h_3 \rightarrow \hat{y}_1, \hat{y}_2, \hat{y}_3 \rightarrow L_1, L_2, L_3 \rightarrow \text{反向传播} \rightarrow \frac{\partial L}{\partial W}, \frac{\partial L}{\partial b} \rightarrow \text{参数更新}$

##### 1.2 这一节的重点不是算具体数字，而是要彻底看懂：

- 前向时每一步算什么
- 反向时梯度按什么顺序传
- 为什么梯度要按时间步累加
- 最后参数到底怎么更新

#### 2、先说明这个示例的模型结构

##### 2.1 我们仍然使用最基础的单层 RNN：

隐藏层前向传播：

$a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h$

$h_t = \tanh(a_t)$

输出层前向传播：

$o_t = W_{hy}h_t + b_y$

$\hat{y}_t = g(o_t)$

其中：

- $x_t$：第 $t$ 个时间步输入
- $h_{t-1}$：前一时刻隐藏状态
- $a_t$：隐藏层线性结果
- $h_t$：当前隐藏状态
- $o_t$：输出层线性结果
- $\hat{y}_t$：当前预测结果

##### 2.2 总时间步设为 3，所以整个序列为：

$x_1,\;x_2,\;x_3$

并且初始隐藏状态设为：

$h_0 = 0$

这是最常见的设定。✅

##### 2.3 总损失定义为三个时间步损失之和：

$L = L_1 + L_2 + L_3$

也可以写成：

$L = \sum_{t=1}^{3} L_t$

这意味着：

每一个时间步都会对最终训练产生贡献。

#### 3、前向传播：先一步一步往前算

##### 3.1 第 1 个时间步

先计算隐藏层线性部分：

$a_1 = W_{xh}x_1 + W_{hh}h_0 + b_h$

因为初始隐藏状态 $h_0 = 0$，所以这里本质上就是：

$a_1 = W_{xh}x_1 + b_h$

然后经过激活函数得到隐藏状态：

$h_1 = \tanh(a_1)$

再计算输出层：

$o_1 = W_{hy}h_1 + b_y$

$\hat{y}_1 = g(o_1)$

最后根据标签 $y_1$ 计算当前损失：

$L_1 = \mathrm{loss}(\hat{y}_1, y_1)$

##### 3.2 第 2 个时间步

这时前一个隐藏状态已经不是 0 了，而是第 1 步算出来的 $h_1$。

所以：

$a_2 = W_{xh}x_2 + W_{hh}h_1 + b_h$

$h_2 = \tanh(a_2)$

$o_2 = W_{hy}h_2 + b_y$

$\hat{y}_2 = g(o_2)$

$L_2 = \mathrm{loss}(\hat{y}_2, y_2)$

这里最关键的一点是：

第 2 步不仅用了当前输入 $x_2$，还用了第 1 步传来的状态 $h_1$。

这就是 RNN “有记忆”的来源。🧠

##### 3.3 第 3 个时间步

同理：

$a_3 = W_{xh}x_3 + W_{hh}h_2 + b_h$

$h_3 = \tanh(a_3)$

$o_3 = W_{hy}h_3 + b_y$

$\hat{y}_3 = g(o_3)$

$L_3 = \mathrm{loss}(\hat{y}_3, y_3)$

##### 3.4 得到整个序列总损失

把三个时间步加起来：

$L = L_1 + L_2 + L_3$

到这里，前向传播结束。

#### 4、把整个 3 步前向链写成展开图

为了后面做反向传播，我们先把这 3 步展开：

##### 4.1 时间步 1

$x_1 \rightarrow a_1 \rightarrow h_1 \rightarrow o_1 \rightarrow \hat{y}_1 \rightarrow L_1$

##### 4.2 时间步 2

$x_2 \rightarrow a_2 \rightarrow h_2 \rightarrow o_2 \rightarrow \hat{y}_2 \rightarrow L_2$

##### 4.3 时间步 3

$x_3 \rightarrow a_3 \rightarrow h_3 \rightarrow o_3 \rightarrow \hat{y}_3 \rightarrow L_3$

##### 4.4 同时还有时间连接：

$h_0 \rightarrow a_1 \rightarrow h_1 \rightarrow a_2 \rightarrow h_2 \rightarrow a_3 \rightarrow h_3$

所以完整理解应该是：

- 每个时间步内部，像一个小型 MLP
- 时间步之间，又通过隐藏状态串联起来


#### 5、开始反向传播：先从最后一个时间步往回

##### 5.1 为什么反向传播一定从第 3 步开始？

因为总损失是：

$L = L_1 + L_2 + L_3$

在时间展开图里，最靠后的节点是 $L_3$，所以反向传播自然从最后往前走：

$L_3 \rightarrow o_3 \rightarrow h_3 \rightarrow a_3 \rightarrow h_2 \rightarrow a_2 \rightarrow h_1 \rightarrow a_1$

这就是 $BPTT$：$Backpropagation\ Through\ Time$。⏪

##### 5.2 第 3 步先求输出层误差项

定义输出层误差项：

$\delta_3^o = \frac{\partial L_3}{\partial o_3}$

它表示：

第 3 个时间步损失，对第 3 个时间步输出层线性结果的梯度。

如果输出层是 $softmax$，损失是 $cross\ entropy$，则常见结果是：

$\delta_3^o = \hat{y}_3 - y_3$

##### 5.3 用上一步的误差项求输出层参数梯度贡献

第 3 步对输出层参数的贡献为：

$\frac{\partial L_3}{\partial W_{hy}} = \delta_3^o h_3^T$

$\frac{\partial L_3}{\partial b_y} = \delta_3^o$

##### 5.4 第 3 步输出层传回隐藏状态

由：

$o_3 = W_{hy}h_3 + b_y$

得到：

$\frac{\partial L_3}{\partial h_3} = W_{hy}^T \delta_3^o$

因为第 3 步已经是最后一步了，它后面没有 $h_4$，所以这里：

$h_3$ 的梯度只来自当前输出层。

##### 5.5 第 3 步继续穿过激活函数

由于：

$h_3 = \tanh(a_3)$

所以隐藏层误差项为：

$\delta_3^h = \frac{\partial L}{\partial a_3} = \left(W_{hy}^T\delta_3^o\right)\odot (1-h_3^2)$

这一步表示：

第 3 步的误差信号已经从输出层传回到了隐藏层线性部分 $a_3$。

##### 5.6 用第 3 步隐藏层误差项求隐藏层参数梯度贡献

第 3 步对隐藏层参数的贡献为：

$\frac{\partial L}{\partial W_{xh}}\bigg|_{t=3} = \delta_3^h x_3^T$

$\frac{\partial L}{\partial W_{hh}}\bigg|_{t=3} = \delta_3^h h_2^T$

$\frac{\partial L}{\partial b_h}\bigg|_{t=3} = \delta_3^h$

#### 6、第 2 个时间步：开始出现“梯度汇合”

##### 6.1 第 2 步为什么比第 3 步复杂？

因为 $h_2$ 不仅影响当前时刻的输出 $o_2$，还影响下一个时间步的隐藏层 $a_3$。

也就是说，$h_2$ 有两条下游路径：

当前输出路径：$h_2 \rightarrow o_2 \rightarrow L_2$

未来时间路径：$h_2 \rightarrow a_3 \rightarrow h_3 \rightarrow L_3$

所以第 2 步的梯度会“汇合”。⭐

##### 6.2 先写第 2 步输出层误差项

$\delta_2^o = \frac{\partial L_2}{\partial o_2}$

同样，如果是 $softmax + cross\ entropy$：

$\delta_2^o = \hat{y}_2 - y_2$

##### 6.3 第 2 步对输出层参数的梯度贡献

$\frac{\partial L_2}{\partial W_{hy}} = \delta_2^o h_2^T$

$\frac{\partial L_2}{\partial b_y} = \delta_2^o$

##### 6.4 第 2 步总的隐藏状态梯度

这一步非常关键。

对 $h_2$ 来说，它的总梯度不是只有当前输出层这一条，而是两部分相加：

$\frac{\partial L}{\partial h_2} = W_{hy}^T\delta_2^o + W_{hh}^T\delta_3^h$

其中：

- $W_{hy}^T\delta_2^o$：来自当前输出层
- $W_{hh}^T\delta_3^h$：来自未来时间步第 3 步

这就是 BPTT 的核心思想。🔥

##### 6.5 穿过第 2 步激活函数

$\delta_2^h = \left(W_{hy}^T\delta_2^o + W_{hh}^T\delta_3^h\right)\odot (1-h_2^2)$

##### 6.6 第 2 步对隐藏层参数的梯度贡献

$\frac{\partial L}{\partial W_{xh}}\bigg|_{t=2} = \delta_2^h x_2^T$

$\frac{\partial L}{\partial W_{hh}}\bigg|_{t=2} = \delta_2^h h_1^T$

$\frac{\partial L}{\partial b_h}\bigg|_{t=2} = \delta_2^h$


#### 7、第 1 个时间步：最早时刻的梯度最“长”

##### 7.1 第 1 步输出层误差项

$\delta_1^o = \frac{\partial L_1}{\partial o_1}$

如果是 $softmax + cross\ entropy$：

$\delta_1^o = \hat{y}_1 - y_1$

##### 7.2 第 1 步对输出层参数的梯度贡献

$\frac{\partial L_1}{\partial W_{hy}} = \delta_1^o h_1^T$

$\frac{\partial L_1}{\partial b_y} = \delta_1^o$

##### 7.3 第 1 步总的隐藏状态梯度

现在 $h_1$ 也有两部分梯度来源：

- 当前时刻输出层
- 第 2 步从未来传回来的梯度（包含了第 3 步的影响）

所以：

$\frac{\partial L}{\partial h_1} = W_{hy}^T\delta_1^o + W_{hh}^T\delta_2^h$

##### 7.4 穿过激活函数得到第 1 步隐藏层误差项

$\delta_1^h = \left(W_{hy}^T\delta_1^o + W_{hh}^T\delta_2^h\right)\odot (1-h_1^2)$

##### 7.5 第 1 步对隐藏层参数的梯度贡献

$\frac{\partial L}{\partial W_{xh}}\bigg|_{t=1} = \delta_1^h x_1^T$

$\frac{\partial L}{\partial W_{hh}}\bigg|_{t=1} = \delta_1^h h_0^T$

$\frac{\partial L}{\partial b_h}\bigg|_{t=1} = \delta_1^h$


#### 8、把 3 个时间步的梯度全部累加

##### 8.1 为什么一定要累加？

因为 RNN 的参数是共享的。

同一个 $W_{xh}$、$W_{hh}$、$W_{hy}$ 在三个时间步都被重复使用。

所以总梯度必须把每一步的贡献相加。

##### 8.2 输出层参数总梯度

$\frac{\partial L}{\partial W_{hy}} = \delta_1^o h_1^T + \delta_2^o h_2^T + \delta_3^o h_3^T$

$\frac{\partial L}{\partial b_y} = \delta_1^o + \delta_2^o + \delta_3^o$

##### 8.3 输入到隐藏层权重总梯度

$\frac{\partial L}{\partial W_{xh}} = \delta_1^h x_1^T + \delta_2^h x_2^T + \delta_3^h x_3^T$

##### 8.4 隐藏到隐藏层权重总梯度

$\frac{\partial L}{\partial W_{hh}} = \delta_1^h h_0^T + \delta_2^h h_1^T + \delta_3^h h_2^T$

##### 8.5 隐藏层偏置总梯度

$\frac{\partial L}{\partial b_h} = \delta_1^h + \delta_2^h + \delta_3^h$

#### 9、最后一步：梯度下降更新参数

##### 9.1 为什么前面所有推导最后都要落到这一步？

因为训练神经网络的最终目的，不是只把梯度算出来，而是要用梯度去更新参数，让损失变小。

假设学习率为 $\eta$，那么最基础的梯度下降更新规则是：

$W \leftarrow W - \eta \frac{\partial L}{\partial W}$

$b \leftarrow b - \eta \frac{\partial L}{\partial b}$

##### 9.2 应用到 RNN 的各组参数

输出层权重更新：

$W_{hy} \leftarrow W_{hy} - \eta \frac{\partial L}{\partial W_{hy}}$

输出层偏置更新：

$b_y \leftarrow b_y - \eta \frac{\partial L}{\partial b_y}$

隐藏层输入权重更新：

$W_{xh} \leftarrow W_{xh} - \eta \frac{\partial L}{\partial W_{xh}}$

循环权重更新：

$W_{hh} \leftarrow W_{hh} - \eta \frac{\partial L}{\partial W_{hh}}$

隐藏层偏置更新：

$b_h \leftarrow b_h - \eta \frac{\partial L}{\partial b_h}$

##### 9.3 更新完之后会发生什么？

参数更新后，下一轮再输入同样的序列：

$x_1,\;x_2,\;x_3$

前向传播得到的：

- $h_1, h_2, h_3$
- $\hat{y}_1, \hat{y}_2, \hat{y}_3$
- $L_1, L_2, L_3$

通常都会和上一轮略有不同。

如果训练方向是对的，那么总损失：

$L = L_1 + L_2 + L_3$

会逐渐下降。📉

#### 10、把整个流程压缩成一条完整主线

##### 10.1 前向传播阶段

按时间顺序：

$x_1 \rightarrow h_1 \rightarrow \hat{y}_1 \rightarrow L_1$

$x_2, h_1 \rightarrow h_2 \rightarrow \hat{y}_2 \rightarrow L_2$

$x_3, h_2 \rightarrow h_3 \rightarrow \hat{y}_3 \rightarrow L_3$

最后：

$L = L_1 + L_2 + L_3$

##### 10.2 反向传播阶段

按时间逆序：

先从第 3 步开始：

$L_3 \rightarrow o_3 \rightarrow h_3 \rightarrow a_3$

再传给第 2 步：

$L_2 \rightarrow o_2 \rightarrow h_2 \rightarrow a_2$

同时接收第 3 步从未来传回来的梯度。

最后传到第 1 步：

$L_1 \rightarrow o_1 \rightarrow h_1 \rightarrow a_1$

同时接收第 2 步从未来传回来的梯度。

##### 10.3 梯度累加阶段

把 3 个时间步对共享参数的贡献全部累加：

- 累加到 $W_{hy}$
- 累加到 $b_y$
- 累加到 $W_{xh}$
- 累加到 $W_{hh}$
- 累加到 $b_h$

##### 10.4 参数更新阶段

利用梯度下降：

$\theta \leftarrow \theta - \eta \nabla_\theta L$

完成本轮训练。